# Exploration: RTE `total_forecast`

Notebook d'exploration pour:
- générer un token OAuth RTE depuis un `Basic` base64 éditable,
- appeler l'endpoint `generation_forecast/v3/total_forecast`,
- sur la période **de deux jours autours de la date actuelle** en **heure française** (`Europe/Paris`).


In [1]:
from __future__ import annotations

import datetime as dt
from zoneinfo import ZoneInfo

from src.rte.rte_client import (
    RteAuthConfig,
    RteBalancingEnergyClient,
    RteConsumptionClient,
    RteGenerationForecastClient,
)

from src.rte.consumption_api.rte_consumption_models import ShortTermQueryType


In [2]:
# A MODIFIER: colle ici la valeur base64 de "client_id:client_secret"
RTE_BASIC_AUTH_B64 = "MTg5OTg3ZmYtMmQzYS00YmYzLWFjNGMtMTA0OGZkMzk3YWU4OjBlMmQyY2UzLTU2Y2EtNGNmZS1hMTlhLWYwZDg0NzcxOGU4Zg=="

PARIS_TZ = ZoneInfo("Europe/Paris")

# Optionnel: types de forecast à demander (laisser [] pour ne pas envoyer le paramètre)
TOTAL_FORECAST_TYPES: list[str] = ["D-1", "ID"]

today = dt.datetime.now(PARIS_TZ)

start_date_paris = today - dt.timedelta(days=2)
end_date_paris = today + dt.timedelta(days=2)

print("today:", today.isoformat())
print("start_date_paris:", start_date_paris.isoformat())
print("end_date_paris:  ", end_date_paris.isoformat())


today: 2026-05-04T12:35:26.342001+02:00
start_date_paris: 2026-05-02T12:35:26.342001+02:00
end_date_paris:   2026-05-06T12:35:26.342001+02:00


In [3]:
if "PASTE_BASE64" in RTE_BASIC_AUTH_B64:
    raise ValueError("Renseigne d'abord RTE_BASIC_AUTH_B64 avec ta vraie valeur base64.")

auth_config = RteAuthConfig(basic_authorization_b64=RTE_BASIC_AUTH_B64)
print("Auth config RTE prêt (token géré automatiquement par les clients).")


Auth config RTE prêt (token géré automatiquement par les clients).


In [4]:
auth_config


RteAuthConfig(access_token=None, client_id=None, client_secret=None, basic_authorization_b64=SecretStr('**********'), token_url='https://digital.iservices.rte-france.com/token/oauth/', token_timeout_seconds=20.0)

In [5]:
async with RteGenerationForecastClient(auth=auth_config) as client:
    total_forecast_response = await client.get_total_forecast(
        forecast_types=TOTAL_FORECAST_TYPES or None,
        start_date=start_date_paris,
        end_date=end_date_paris,
    )

total_forecast_payload = total_forecast_response.model_dump(mode="json")
print("Keys:", list(total_forecast_payload.keys()))


Keys: ['total_forecast']


In [6]:
import sys
sys.getsizeof(total_forecast_payload) # returns it in bytes.

184

In [7]:
total_forecast_payload

{'total_forecast': [{'type': None,
   'start_date': '2026-05-02T10:35:34Z',
   'end_date': '2026-05-06T10:35:34Z',
   'values': [],
   'points': [{'quantity': 46936, 'start_date': '2026-05-02T10:45:00Z'},
    {'quantity': 47196, 'start_date': '2026-05-02T11:00:00Z'},
    {'quantity': 47161, 'start_date': '2026-05-02T11:15:00Z'},
    {'quantity': 47332, 'start_date': '2026-05-02T11:30:00Z'},
    {'quantity': 47257, 'start_date': '2026-05-02T11:45:00Z'},
    {'quantity': 46384, 'start_date': '2026-05-02T12:00:00Z'},
    {'quantity': 46484, 'start_date': '2026-05-02T12:15:00Z'},
    {'quantity': 46447, 'start_date': '2026-05-02T12:30:00Z'},
    {'quantity': 46518, 'start_date': '2026-05-02T12:45:00Z'},
    {'quantity': 46850, 'start_date': '2026-05-02T13:00:00Z'},
    {'quantity': 46079, 'start_date': '2026-05-02T13:15:00Z'},
    {'quantity': 46953, 'start_date': '2026-05-02T13:30:00Z'},
    {'quantity': 46511, 'start_date': '2026-05-02T13:45:00Z'},
    {'quantity': 46546, 'start_date': '

In [8]:
# Apercu rapide des donnees retournees
series = total_forecast_payload["total_forecast"]
print("Nombre de series:", len(series))

if series:
    first_series = series[0]["points"]
    # print("Type:", first_series.get("type"))
    # values = first_series.get("values", [])
    print("Nombre de points:", len(first_series))
    print("Premier point:", first_series[0] if first_series else None)


Nombre de series: 1
Nombre de points: 383
Premier point: {'quantity': 46936, 'start_date': '2026-05-02T10:45:00Z'}


In [9]:
today.strftime("%Y-%m-%d")

tomorrow = today + dt.timedelta(days=1)
tomorrow.strftime("%Y-%m-%d")

'2026-05-05'

In [10]:
# Montrer valeurs pour le jour en cours
tomorrow = today + dt.timedelta(days=1)

for dico in total_forecast_payload["total_forecast"][0]["points"]:
    if (dico["start_date"] >= today.strftime("%Y-%m-%d")) & (dico["start_date"] < tomorrow.strftime("%Y-%m-%d")):
        print(dico["start_date"])
        print(dico["quantity"])

# attention, c'est une erreur ici de ne pas traduire en temps Europe/Paris avant de filter

2026-05-04T00:00:00Z
50701
2026-05-04T00:15:00Z
50459
2026-05-04T00:30:00Z
50155
2026-05-04T00:45:00Z
49978
2026-05-04T01:00:00Z
49688
2026-05-04T01:15:00Z
49591
2026-05-04T01:30:00Z
49556
2026-05-04T01:45:00Z
49454
2026-05-04T02:00:00Z
49381
2026-05-04T02:15:00Z
49481
2026-05-04T02:30:00Z
49436
2026-05-04T02:45:00Z
49482
2026-05-04T03:00:00Z
49971
2026-05-04T03:15:00Z
49916
2026-05-04T03:30:00Z
50747
2026-05-04T03:45:00Z
50979
2026-05-04T04:00:00Z
51619
2026-05-04T04:15:00Z
52327
2026-05-04T04:30:00Z
53522
2026-05-04T04:45:00Z
54150
2026-05-04T05:00:00Z
54143
2026-05-04T05:15:00Z
55063
2026-05-04T05:30:00Z
56174
2026-05-04T05:45:00Z
56764
2026-05-04T06:00:00Z
57773
2026-05-04T06:15:00Z
58047
2026-05-04T06:30:00Z
57982
2026-05-04T06:45:00Z
57631
2026-05-04T07:00:00Z
57694
2026-05-04T07:15:00Z
57786
2026-05-04T07:30:00Z
57281
2026-05-04T07:45:00Z
57231
2026-05-04T08:00:00Z
57582
2026-05-04T08:15:00Z
58104
2026-05-04T08:30:00Z
59022
2026-05-04T08:45:00Z
59283
2026-05-04T09:00:00Z
59813
2

In [11]:
# Extra: lire la timezone de dico["start_date"] puis convertir vers PARIS_TZ

raw_start_time = dico["start_date"]
parsed_start_time = dt.datetime.fromisoformat(raw_start_time)

source_tz = parsed_start_time.tzinfo
paris_time = parsed_start_time.astimezone(PARIS_TZ)

print("Timezone source:", source_tz)
print("Date convertie Paris:", paris_time.isoformat())
print(dico["start_date"])

Timezone source: UTC
Date convertie Paris: 2026-05-06T12:15:00+02:00
2026-05-06T10:15:00Z


CCL : 
Je lui ai demandé à partir de 2026-03-24T00:00:00+01:00
il m'a renvoyé à partir de 2026-03-23T23:00:00Z, soit 2026-03-23T00:00:00+01:00
C'est normal.
c'est juste qu'il faut convertir à nouveau en temps europe/paris avant de faire des tris


Fait aussi l'appel aux autres API, pour avoir le poids des réponses:


Short term consumption

In [12]:
start_date_paris = dt.datetime(2026, 3, 24, 0, 0, 0, tzinfo=PARIS_TZ)
end_date_paris = dt.datetime(2026, 3, 28, 23, 59, 59, tzinfo=PARIS_TZ)

short_term_types = [ShortTermQueryType.DAY_AHEAD]

async with RteConsumptionClient(auth=auth_config) as client:
    short_term_response = await client.get_short_term(
        types=short_term_types,
        start_date=start_date_paris,
        end_date=end_date_paris,
    )

short_term_payload = short_term_response.model_dump(mode="json")
print("Keys:", list(short_term_payload.keys()))


Keys: ['short_term']


In [13]:
sys.getsizeof(short_term_payload)

184

In [14]:
short_term_payload

{'short_term': [{'type': 'D-1',
   'start_date': '2026-03-24T00:00:00+01:00',
   'end_date': '2026-03-28T00:00:00+01:00',
   'values': [{'start_date': '2026-03-24T00:00:00+01:00',
     'end_date': '2026-03-24T00:15:00+01:00',
     'updated_date': '2026-03-24T23:57:43+01:00',
     'value': 48604},
    {'start_date': '2026-03-24T00:15:00+01:00',
     'end_date': '2026-03-24T00:30:00+01:00',
     'updated_date': '2026-03-24T23:57:43+01:00',
     'value': 48108},
    {'start_date': '2026-03-24T00:30:00+01:00',
     'end_date': '2026-03-24T00:45:00+01:00',
     'updated_date': '2026-03-23T23:55:19+01:00',
     'value': 51200},
    {'start_date': '2026-03-24T00:45:00+01:00',
     'end_date': '2026-03-24T01:00:00+01:00',
     'updated_date': '2026-03-23T23:55:19+01:00',
     'value': 51292},
    {'start_date': '2026-03-24T01:00:00+01:00',
     'end_date': '2026-03-24T01:15:00+01:00',
     'updated_date': '2026-03-23T23:55:19+01:00',
     'value': 49400},
    {'start_date': '2026-03-24T01:15:0

PRE

In [15]:
start_date_paris = dt.datetime(2026, 3, 24, 0, 0, 0, tzinfo=PARIS_TZ)
end_date_paris = dt.datetime(2026, 3, 28, 23, 59, 59, tzinfo=PARIS_TZ)

async with RteBalancingEnergyClient(auth=auth_config) as client:
    imbalance_data_response = await client.get_imbalance_data(
        start_date=start_date_paris,
        end_date=end_date_paris,
    )

imbalance_data_payload = imbalance_data_response.model_dump(mode="json")
print("Keys:", list(imbalance_data_payload.keys()))


Keys: ['imbalance_data']


In [16]:
# # pour mesurer le poids de ces données
# import csv
# with open('imbalance_data_payload.csv', 'w') as csv_file:  
#     writer = csv.writer(csv_file)
#     for key, value in imbalance_data_payload.items():
#        writer.writerow([key, value])

# with open('short_term_payload.csv', 'w') as csv_file:  
#     writer = csv.writer(csv_file)
#     for key, value in short_term_payload.items():
#        writer.writerow([key, value])

# with open('total_forecast_payload.csv', 'w') as csv_file:  
#     writer = csv.writer(csv_file)
#     for key, value in total_forecast_payload.items():
#        writer.writerow([key, value])

In [17]:
imbalance_data_payload

{'imbalance_data': [{'start_date': '2026-03-24T00:00:00+01:00',
   'end_date': '2026-03-28T23:45:00+01:00',
   'resolution': 'PT15M',
   'values': [{'start_date': '2026-03-28T23:30:00+01:00',
     'end_date': '2026-03-28T23:45:00+01:00',
     'imbalance': 380.92,
     'system_trend': 'BAISSE',
     'positive_imbalance_settlement_price': 41.75,
     'negative_imbalance_settlement_price': 51.77,
     'missing_data_list': 'N/A',
     'updated_date': '2026-03-28T23:47:48+01:00'},
    {'start_date': '2026-03-28T23:15:00+01:00',
     'end_date': '2026-03-28T23:30:00+01:00',
     'imbalance': 458.65,
     'system_trend': 'BAISSE',
     'positive_imbalance_settlement_price': -37.98,
     'negative_imbalance_settlement_price': -30.62,
     'missing_data_list': 'N/A',
     'updated_date': '2026-03-29T08:28:25+02:00'},
    {'start_date': '2026-03-28T23:00:00+01:00',
     'end_date': '2026-03-28T23:15:00+01:00',
     'imbalance': 330.45,
     'system_trend': 'BAISSE',
     'positive_imbalance_sett